In [0]:
from pyspark.sql import SparkSession, functions as f

#Reading Hospital A departments data 
df_hosa = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/hosa/providers")

#Reading Hospital B departments data 
df_hosb = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/hosb/providers")

#union two departments dataframes
df_merged = df_hosa.unionByName(df_hosb)
display(df_merged)

df_merged.createOrReplaceTempView("providers")


ProviderID,FirstName,LastName,Specialization,DeptID,NPI,datasource
H1-PROV0001,Brandon,Harper,Oncology,DEPT018,3086631719,hos-a
H1-PROV0002,Luke,Clark,Emergency Medicine,DEPT006,1265568618,hos-a
H1-PROV0003,Brandon,Austin,Pediatrics,DEPT014,3152052791,hos-a
H1-PROV0004,Anthony,Jones,Radiology,DEPT011,8195895822,hos-a
H1-PROV0005,Courtney,Flores,Anesthesiology,DEPT009,4159583915,hos-a
H1-PROV0006,Ricky,Johnson,Neurology,DEPT017,7386802836,hos-a
H1-PROV0007,John,Carroll,Neurology,DEPT014,9500252998,hos-a
H1-PROV0008,William,Stout,Emergency Medicine,DEPT005,3263108127,hos-a
H1-PROV0009,Kelly,Cardenas,Neurology,DEPT005,971099357,hos-a
H1-PROV0010,Dawn,Valdez,Psychiatry,DEPT008,8973799839,hos-a


In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.providers (
ProviderID string,
FirstName string,
LastName string,
Specialization string,
DeptID string,
NPI long,
datasource string,
is_quarantined boolean
)
USING DELTA;

In [0]:
%sql
truncate table silver.providers

In [0]:
%sql 
insert into silver.providers
select 
distinct
ProviderID,
FirstName,
LastName,
Specialization,
DeptID,
try_cast(NPI as INT) NPI,
datasource,
    CASE 
        WHEN ProviderID IS NULL OR DeptID IS NULL THEN TRUE
        ELSE FALSE
    END AS is_quarantined
from providers

num_affected_rows,num_inserted_rows
55,55


In [0]:
df_sil_providers = spark.read.format("delta").table("silver.providers")

# Define your ADLS Gen2 path
adls_path = "abfss://silver@ttadlsjcrh.dfs.core.windows.net/"

# Write the DataFrame as a Delta table
df_sil_providers.write.format("delta").mode("overwrite").save(adls_path + "/providers")